<a href="https://colab.research.google.com/github/adlihs/PyTorch/blob/main/Building_a_CNN_for_Nature_Classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

* **Prepare a Diverse Dataset**: Load and transform a specialized subset of images for your multi-class nature classifier.

* **Build a CNN Architecture**: Define a complete CNN from scratch, combining convolutional, pooling, and fully connected layers to create a powerful feature extractor.

* **Train a Prototype Model**: Follow a realistic workflow by first training your model on a smaller, 9-class subset to build a working prototype and establish a performance baseline.

* **Scale Up and Diagnose Challenges**: Train the full model on all 15 classes and analyze the results to identify common machine learning challenges like overfitting.

# Imports

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
from torch.utils.data import DataLoader

In [2]:
# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cpu


## helper utils

In [3]:
import copy
import os
import random
from collections import defaultdict

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import torch
import torchvision
from torch.utils.data import DataLoader

In [4]:
def load_cifar100_subset(target_classes, train_transform, val_transform, root='./cifar_100'):
    """
    Loads and filters the CIFAR-100 dataset to include only specified target classes.

    This function first checks for a local copy of the CIFAR-100 dataset and
    downloads it if not found. It then filters both the training and test sets
    to retain only the images and labels corresponding to the classes specified
    in `target_classes`. The labels are remapped to be contiguous from 0.

    Args:
        target_classes: A list of class name strings to be included in the dataset subset.
        train_transform: A torchvision transform to be applied to the training dataset images.
        val_transform: A torchvision transform to be applied to the test/val dataset images.
        root: The root directory where the dataset is stored or will be downloaded.

    Returns:
        A tuple containing the filtered training dataset and the filtered test dataset.
        Returns (None, None) if a specified target class is not found.
    """
    # Construct the path to the CIFAR-100 dataset directory.
    cifar100_path = os.path.join(root, 'cifar-100-python')
    # Check if the dataset directory exists locally.
    if os.path.isdir(cifar100_path):
        print(f"Dataset found in '{root}'. Loading from local files.")
    # If not found, inform the user that it will be downloaded.
    else:
        print(f"Dataset not found in '{root}'. Downloading...")

    # Load the full CIFAR-100 training dataset.
    train_dataset_full = torchvision.datasets.CIFAR100(
        root=root,
        train=True,
        download=True,
        transform=train_transform
    )

    # Load the full CIFAR-100 test dataset.
    test_dataset_full = torchvision.datasets.CIFAR100(
        root=root,
        train=False,
        download=True,
        transform=val_transform
    )
    print("Dataset loaded successfully.")

    # Get the list of all class names from the dataset.
    all_classes = train_dataset_full.classes
    try:
        # Get the original integer indices for the target class names.
        target_indices = [all_classes.index(cls) for cls in target_classes]
    # Handle the case where a specified class name is not in the dataset.
    except ValueError as e:
        print(f"Error: One of the target classes not found in CIFAR-100. {e}")
        return None, None

    # Create a mapping from the original class indices to new, contiguous indices (0, 1, 2, ...).
    label_map = {old_label: new_label for new_label, old_label in enumerate(target_indices)}

    # Define a helper function to filter a dataset based on the target classes.
    def _filter_dataset(dataset):
        # Convert the list of targets to a NumPy array for efficient boolean indexing.
        targets_np = np.array(dataset.targets)
        # Create a boolean mask to identify which samples belong to the target classes.
        indices_to_keep = np.isin(targets_np, target_indices)

        # Filter the dataset's image data using the boolean mask.
        dataset.data = dataset.data[indices_to_keep]

        # Get the original labels of the samples that are being kept.
        original_targets_to_keep = targets_np[indices_to_keep]
        # Remap the original labels to the new contiguous labels.
        dataset.targets = [label_map[target] for target in original_targets_to_keep]

        # Update the dataset's class list to only include the target classes.
        dataset.classes = target_classes
        return dataset

    print(f"\nFiltering for {len(target_classes)} classes...")
    # Apply the filtering logic to the full training dataset.
    train_dataset_subset = _filter_dataset(train_dataset_full)
    # Apply the filtering logic to the full test dataset.
    test_dataset_subset = _filter_dataset(test_dataset_full)
    print("Filtering complete. Returning training and validation datasets.")

    # Return the filtered training and test subsets.
    return train_dataset_subset, test_dataset_subset

In [5]:
def visualise_images(dataset, grid):
    """
    Displays a grid of images from a dataset, with one random image per class.

    Args:
        dataset: The dataset object containing the images and labels.
        grid (tuple): A tuple specifying the number of rows and columns for the image grid.
    """

    # Create a shallow copy of the dataset to avoid modifying the original
    dataset_copy = copy.copy(dataset)
    # Set the transform on the copied dataset to convert images to tensors
    dataset_copy.transform = torchvision.transforms.ToTensor()

    # Create a DataLoader to handle batching and shuffling of the data
    loader = DataLoader(dataset_copy, batch_size=64, shuffle=True)

    # Unpack the grid dimensions from the input tuple
    rows, cols = grid
    # Calculate the total number of images to display in the grid
    num_images_to_show = rows * cols

    # Get the dataset object from the DataLoader
    dataset_to_show = loader.dataset

    # Create a dictionary to store lists of indices for each class
    class_indices = defaultdict(list)
    # Iterate through the dataset to populate the class_indices dictionary
    for idx, target in enumerate(dataset_to_show.targets):
        class_indices[target].append(idx)

    # Get the list of class names from the dataset
    class_names = dataset_to_show.classes

    # Create a figure and a set of subplots for the grid layout
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 2, rows * 2))

    # Iterate over each subplot in the grid
    for i, ax in enumerate(axes.flat):
        # If the current index is out of bounds, turn off the subplot axis
        if i >= num_images_to_show or i >= len(class_names):
            ax.axis('off')
            continue

        # Set the class label based on the current iteration index
        class_label = i

        # Get the list of image indices for the current class
        indices_for_class = class_indices[class_label]
        # If there are no images for this class, turn off the subplot axis
        if not indices_for_class:
            ax.axis('off')
            continue

        # Choose a random image index from the list for the current class
        random_image_index = random.choice(indices_for_class)

        # Retrieve the image tensor and its corresponding label from the dataset
        image_tensor, _ = dataset_to_show[random_image_index]

        # Convert the tensor to a NumPy array and transpose dimensions for display
        img_to_display = image_tensor.numpy().transpose((1, 2, 0))

        # Get the name of the class corresponding to the class label
        class_name = class_names[class_label]

        # Display the image on the current subplot
        ax.imshow(img_to_display)

        # Set the title of the subplot to the capitalized class name
        ax.set_title(class_name.capitalize(), fontsize=16)
        # Turn off the axis for a cleaner look
        ax.axis('off')

    # Adjust subplot parameters for a tight layout
    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    # Display the plot
    plt.show()

    # Clean up the copied dataset to free up memory
    del dataset_copy

In [6]:
def plot_training_metrics(metrics):
    """
    Plots the training and validation metrics from a model training process.

    This function generates two side-by-side plots:
    1. Training Loss vs. Validation Loss.
    2. Validation Accuracy.

    Args:
        metrics (list): A list or tuple containing three lists:
                        [train_losses, val_losses, val_accuracies].
    """
    # Unpack the metrics into their respective lists
    train_losses, val_losses, val_accuracies = metrics

    # Determine the number of epochs from the length of the training losses list
    num_epochs = len(train_losses)
    # Create a 1-indexed range of epoch numbers for the x-axis
    epochs = range(1, num_epochs + 1)

    # Create a figure and a set of subplots with 1 row and 2 columns
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    # --- Configure the first subplot for training and validation loss ---
    # Select the first subplot
    ax1 = axes[0]
    # Plot training loss data
    ax1.plot(epochs, train_losses, color='#085c75', linewidth=2.5, marker='o', markersize=5, label='Training Loss')
    # Plot validation loss data
    ax1.plot(epochs, val_losses, color='#fa5f64', linewidth=2.5, marker='o', markersize=5, label='Validation Loss')
    # Set the title and axis labels for the loss plot
    ax1.set_title('Training & Validation Loss', fontsize=14)
    ax1.set_xlabel('Epoch', fontsize=12)
    ax1.set_ylabel('Loss', fontsize=12)
    # Display the legend
    ax1.legend()
    # Add a grid for better readability
    ax1.grid(True, linestyle='--', alpha=0.6)

    # --- Configure the second subplot for validation accuracy ---
    # Select the second subplot
    ax2 = axes[1]
    # Plot validation accuracy data
    ax2.plot(epochs, val_accuracies, color='#fa5f64', linewidth=2.5, marker='o', markersize=5, label='Validation Accuracy')
    # Set the title and axis labels for the accuracy plot
    ax2.set_title('Validation Accuracy', fontsize=14)
    ax2.set_xlabel('Epoch', fontsize=12)
    ax2.set_ylabel('Accuracy (%)', fontsize=12)
    # Display the legend
    ax2.legend()
    # Add a grid for better readability
    ax2.grid(True, linestyle='--', alpha=0.6)

    # --- Apply dynamic and consistent styling to both subplots ---
    # Calculate a suitable interval for the x-axis ticks to avoid clutter
    x_interval = (num_epochs - 1) // 10 + 1

    # Loop through each subplot to apply common axis settings
    for ax in axes:
        # Set the y-axis to start at 0 and the x-axis to span the epochs
        ax.set_ylim(bottom=0)
        ax.set_xlim(left=1, right=num_epochs)

        # Set the major tick locator for the x-axis using the dynamic interval
        ax.xaxis.set_major_locator(mticker.MultipleLocator(x_interval))
        # Set the font size for the tick labels on both axes
        ax.tick_params(axis='both', which='major', labelsize=10)

    # Adjust subplot parameters for a tight layout
    plt.tight_layout()
    # Display the plots
    plt.show()

In [7]:
def visualise_predictions(model, data_loader, device, grid):
    """
    Visualizes model predictions on a grid of images from a dataset.

    Args:
        model: The trained PyTorch model to use for predictions.
        data_loader: The PyTorch DataLoader for the dataset.
        device: The device (e.g., 'cpu' or 'cuda') to run the model on.
        grid (tuple): A tuple specifying the number of rows and columns for the image grid.
    """
    # Set the model to evaluation mode
    model.eval()

    # Get the dataset and class names from the data loader
    dataset = data_loader.dataset
    class_names = dataset.classes

    # Define mean and standard deviation values for de-normalizing the images
    cifar100_mean = np.array([0.5071, 0.4867, 0.4408])
    cifar100_std = np.array([0.2675, 0.2565, 0.2761])

    # Create a dictionary to store lists of indices for each class
    class_indices = defaultdict(list)
    # Iterate through the dataset to populate the class_indices dictionary
    for idx, target in enumerate(dataset.targets):
        class_indices[target].append(idx)

    # Unpack the grid dimensions
    rows, cols = grid
    # Calculate the total number of images to display
    num_images_to_show = rows * cols

    # Create a figure and a set of subplots
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 2, rows * 2))
    # Adjust the spacing between subplots
    plt.subplots_adjust(wspace=0.3, hspace=0.8)

    # Iterate over each subplot in the grid
    for i, ax in enumerate(axes.flat):
        # If the current index is out of bounds, turn off the subplot axis
        if i >= num_images_to_show or i >= len(class_names):
            ax.axis('off')
            continue

        # Set the class label based on the current iteration index
        class_label = i

        # Get the list of image indices for the current class
        indices_for_class = class_indices[class_label]
        # If there are no images for this class, turn off the subplot axis
        if not indices_for_class:
            ax.axis('off')
            continue

        # Choose a random image index from the list for the current class
        random_image_index = random.choice(indices_for_class)
        # Retrieve the image tensor and its true label
        image_tensor, true_label = dataset[random_image_index]

        # Add a batch dimension and move the tensor to the specified device
        image_batch = image_tensor.unsqueeze(0).to(device)

        # Disable gradient calculations for inference
        with torch.no_grad():
            # Get model predictions
            output = model(image_batch)
            # Find the index of the highest score, which is the predicted class
            _, predicted_index = torch.max(output, 1)

        # Extract the predicted label as a Python number
        predicted_label = predicted_index.item()

        # Convert tensor to a NumPy array and transpose dimensions for display
        img_np = image_tensor.cpu().numpy().transpose((1, 2, 0))
        # De-normalize the image using the predefined mean and std
        denormalized_img = cifar100_std * img_np + cifar100_mean
        # Clip the pixel values to the valid range [0, 1]
        clipped_img = np.clip(denormalized_img, 0, 1)

        # Get the string names for the true and predicted labels
        true_name = class_names[true_label]
        predicted_name = class_names[predicted_label]

        # Set the title color to green for correct predictions and red for incorrect ones
        title_color = 'green' if true_label == predicted_label else 'red'

        # Display the image
        ax.imshow(clipped_img)
        # Set the title with true and predicted labels
        ax.set_title(f"True: {true_name.capitalize()}\nPred: {predicted_name.capitalize()}",
                     color=title_color, fontsize=10, pad=5)
        # Turn off the axis
        ax.axis('off')

    # Adjust subplot parameters for a tight layout
    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    # Show the final plot
    plt.show()

# Preparing the nature Dataset
This dataset is a fantastic resource for computer vision tasks, containing thousands of small, **32x32 color images** that are perfect for training a CNN. It’s a diverse collection, which is exactly what you need for the expanded nature classifier app.

While CIFAR-100 has 100 different classes, you won't need all of them. To meet the new requirements for your app, you'll focus on a curated selection of **15 classes** that fit the theme of a nature classifier. This selection will include flowers, insects and mammals. Specifically, you'll be working with:

* **Flowers**: 'orchid', 'poppy', 'rose', 'sunflower', 'tulip'

* **Mammals**: 'fox', 'porcupine', 'possum', 'raccoon', 'skunk'

* **Insects**: 'bee', 'beetle', 'butterfly', 'caterpillar', 'cockroach'


### Image Transformations

Before you load the dataset, first you need to define the transformation pipelines for it. Since all images in the dataset are already a standard **32x32** size, you don't need to add a resizing step. Your training pipeline will include data augmentation, while both pipelines will convert images to **tensors** and **normalize** them using the standard mean and standard deviation for the CIFAR-100 dataset.

* Define the specific mean and standard deviation for the CIFAR-100 dataset.

In [8]:
cifar100_mean = (0.5071, 0.4867, 0.4408)
cifar100_std = (0.2675, 0.2565, 0.2761)

* Define two separate pipelines using `transforms.Compose`.
    * One for the training set that includes data augmentation and another for the validation set.

In [9]:
# Training set transformation pipeline
train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize(cifar100_mean, cifar100_std)
])

# Validation set transformation pipeline
val_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(cifar100_mean, cifar100_std)
])

# Preparing the Data Pipeline

With your transformations ready, it's time to load the data. To quickly show the butterfly house next door a working prototype, it's a smart strategy to start with a smaller, more manageable dataset. This allows you to test your entire pipeline and build a baseline model without the long wait times required for the full dataset.

Therefore, instead of using all 15 classes at once, you'll begin with a balanced subset of **9 classes** (**3 from each category**). This iterative approach is a common and efficient practice in real-world machine learning.

For this initial prototype, you'll use the following classes:

* **Flowers**: 'orchid', 'poppy', 'sunflower'

* **Mammals**: 'fox', 'raccoon', 'skunk'

* **Insects**: 'butterfly', 'caterpillar', 'cockroach'

* Create a Python list containing the names of the 9 classes you'll use for the initial prototype.

In [10]:
subset_target_classes = [
    # Flowers
    'orchid', 'poppy', 'sunflower',
    # Mammals
    'fox', 'raccoon', 'skunk',
    # Insects
    'butterfly', 'caterpillar', 'cockroach'
]

* Use the `load_cifar100_subset` helper function, passing in your `subset_target_classes` list and both transformation pipelines.
* This function handles the entire loading process: it downloads the full CIFAR-100 dataset, applies your specified transformations, and then filters the result to include only the **9 classes** you selected.
* It returns the final training and validation dataset objects, ready for the next step.

In [ ]:
# Call the helper function to prepare the datasets
train_dataset_proto, val_dataset_proto = load_cifar100_subset(subset_target_classes,
                                                             train_transform, val_transform)



Dataset not found in './cifar_100'. Downloading...


 49%|████▊     | 82.4M/169M [12:27<13:54, 104kB/s] 

* With your `Dataset` objects ready, the final step in the data pipeline is to create `DataLoaders`.

In [ ]:
# Set the number of samples to be processed in each batch
batch_size = 64

# Create a data loader for the training set, with shuffling enabled
train_loader_proto = DataLoader(train_dataset_proto, batch_size=batch_size, shuffle=True)

# Create a data loader for the validation set, without shuffling
val_loader_proto =  DataLoader(val_dataset_proto, batch_size=batch_size, shuffle=False)

# Visualizing the Training Images

With your data pipeline complete, it's always a good idea to look at a few examples from your training set. This helps confirm that your data has been loaded and processed correctly. The following helper function will display a random sample of your training images.

In [ ]:
# Visualize a 3x3 grid of random training images
visualise_images(train_dataset_proto, grid=3,3)

# Building the CNN Architecture

With your data ready, it's time to build the core of your nature classifier. For a complex task like identifying different species in images, the linear layers you've used before aren't enough, as they look at pixels individually without understanding their spatial relationships.

You'll now build a **Convolutional Neural Network (CNN)**, an architecture specifically designed to "see" and recognize patterns, edges, and textures in images through a series of learnable filters. You'll define your model's structure using PyTorch's `nn.Module`, combining several types of layers to create a powerful image classifier.

Here's a breakdown of the key layers you'll use:

**Convolutional Layer (<code>[nn.Conv2d](https://docs.pytorch.org/docs/stable/generated/torch.nn.Conv2d.html)</code>)**
> This is the core building block of a CNN, using learnable filters to scan the image for visual features. The output is a set of "feature maps" that highlight where in the image these patterns appear.
> * `in_channels`: The number of channels from the previous layer; for the first layer, this is 3 for the RGB color channels.
> * `out_channels`: The number of filters the layer will learn, determining the number of output feature maps.
> * `kernel_size`: The dimensions of the filter, such as a 3x3 grid that examines a pixel and its immediate neighbors.
> * `padding`: Adds a border around the image, allowing the kernel to process edge pixels while preserving the image's dimensions.

**ReLU Activation Function (<code>[nn.ReLU](https://docs.pytorch.org/docs/stable/generated/torch.nn.ReLU.html)</code>)**
> An activation function that introduces non-linearity by changing all negative values in the feature maps to zero. This helps the model learn more complex patterns.

**Max Pooling Layer (<code>[nn.MaxPool2d](https://docs.pytorch.org/docs/stable/generated/torch.nn.MaxPool2d.html)</code>)**
> This layer downsamples the feature maps by reducing their height and width, which makes the network more efficient. It slides a window over the feature map and keeps only the single largest value from that window, discarding the rest.
> * `kernel_size`: The size of the window to perform pooling on, such as a 2x2 area.
> * `stride`: The step size the window moves across the image. A stride of 2 with a 2x2 kernel will halve the feature map's dimensions.

**Flatten Layer (<code>[nn.Flatten](https://docs.pytorch.org/docs/stable/generated/torch.nn.Flatten.html)</code>)**
> A utility layer that unrolls the 2D feature maps into a single 1D vector. This is a necessary step to prepare the data for the fully connected linear layers.

**Linear Layer (<code>[nn.Linear](https://docs.pytorch.org/docs/stable/generated/torch.nn.Linear.html)</code>)**
> Also known as a fully connected layer, it performs the final classification. It combines the features learned by the convolutional layers into a final prediction.

**Dropout Layer (<code>[nn.Dropout](https://docs.pytorch.org/docs/stable/generated/torch.nn.Dropout.html)</code>)**
> A regularization technique that helps prevent overfitting by randomly setting a fraction of neuron activations to zero during training. This forces the network to learn more robust features instead of relying too heavily on any single pattern.

In [ ]:
class SimpleCNN (nn.module):
  """
  A simple Convolutional Neural Network model.

  The architecture consists of three convolutional blocks followed by two
  fully connected layers for classification.
  """
  def __init__(self, num_classes):
      """
      Initializes the layers of the neural network.

      Args:
          num_classes: The number of output classes for the final layer.
      """
      # Call the constructor of the parent class (nn.Module)
      super(SimpleCNN, self).__init__()

      # Define the first convolutional block
      self.conv1 = nn.Conv2d(in_channels=3, out_channels=32, kernel_size=3, padding=1)
      self.relu1 = nn.ReLU()
      self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)

      # Define the second convolutional block
      self.conv2 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1)
      self.relu2 = nn.ReLU()
      self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)

      # Define the third convolutional block
      self.conv3 = nn.Conv2d(in_channels=64, out_channels=128, kernel_size=3, padding=1)
      self.relu3 = nn.ReLU()
      self.pool3 = nn.MaxPool2d(kernel_size=2, stride=2)

      # Define the layer to flatten the feature maps
      self.flatten = nn.Flatten()

      # Define the fully connected (dense) layers
      # Input image is 32x32, after 3 pooling layers: 4x4
      self.fc1 = nn.Linear(128 * 4 * 4, 512)
      self.relu4 = nn.ReLU()
      self.dropout = nn.Dropout(0.5)
      self.fc2 = nn.Linear(512, num_classes)


  def forward(self, x):
      """
      Defines the forward pass of the model.

      Args:
          x: The input tensor of shape (batch_size, channels, height, width).

      Returns:
          The output tensor containing the logits for each class.
      """
      # Pass input through the first convolutional block
      x = self.conv1(x)
      x = self.relu1(x)
      x = self.pool1(x)

      # Pass feature maps through the second convolutional block
      x = self.conv2(x)
      x = self.relu2(x)
      x = self.pool2(x)

      # Pass feature maps through the third convolutional block
      x = self.conv3(x)
      x = self.relu3(x)
      x = self.pool3(x)

      # Flatten the output for the fully connected layers
      x = self.flatten(x)

      # Pass the flattened features through the fully connected layers
      x = self.fc1(x)
      x = self.relu4(x)
      x = self.dropout(x)
      x = self.fc2(x)

      # Return the final output logits
      return x





With the `SimpleCNN` architecture defined, the next step is to create an instance of the model for your prototype.

* First, dynamically determine the number of output classes by checking the length of the class list in your `train_dataset_proto`.
* Create an instance of your `SimpleCNN`, passing `num_classes` to its constructor.

In [ ]:
# Get the number of classes
num_classes = len(train_dataset_proto.classes)

# Instantiate the model
prototype_model = SimpleCNN(num_classes)

Before you start training, it's very helpful to visualize how the shape of your data changes as it flows through the CNN. This will confirm that your architecture is set up correctly and show you how the spatial dimensions shrink while the number of channels grows with each convolutional block.

* Define the `print_data_flow` helper function.
    * This function will pass a sample 32x32 color image through your model, layer by layer, printing the tensor's shape at each key step to trace its journey from input to final prediction.

In [ ]:
def print_data_flow(model):
    """
    Prints the shape of a tensor as it flows through each layer of the model.

    Args:
        model: An instance of the PyTorch model to inspect.
    """
    # Create a sample input tensor (batch_size, channels, height, width)
    x = torch.randn(1, 3, 32, 32)

    # Track the tensor shape at each stage
    print(f"Input shape: \t\t{x.shape}")

    # First conv block
    x = model.conv1(x)
    print(f"After conv1: \t\t{x.shape}")
    x = model.relu1(x)
    x = model.pool1(x)
    print(f"After pool1: \t\t{x.shape}")

    # Second conv block
    x = model.conv2(x)
    print(f"After conv2: \t\t{x.shape}")
    x = model.relu2(x)
    x = model.pool2(x)
    print(f"After pool2: \t\t{x.shape}")

    # Third conv block
    x = model.conv3(x)
    print(f"After conv3: \t\t{x.shape}")
    x = model.relu3(x)
    x = model.pool3(x)
    print(f"After pool3: \t\t{x.shape}")

    # Flatten using the model's flatten layer
    x = model.flatten(x)
    print(f"After flatten: \t\t{x.shape}")

    # Fully connected layers
    x = model.fc1(x)
    print(f"After fc1: \t\t{x.shape}")
    x = model.relu4(x)
    x = model.dropout(x)
    x = model.fc2(x)
    print(f"Output shape (fc2): \t{x.shape}")

You can now print a summary of your model and trace the data flow to see it in action.

* Call your helper function to print the tensor's shape at each step.

    * The tensor starts as a `(1, 3, 32, 32)` image. As it passes through the `conv` and `pool` blocks, the number of **channels increases** while the **spatial size is halved** at each step.

    * The final `(1, 128, 4, 4)` feature map is **flattened** into a 1D vector to be processed by the linear layers. The model's final **output shape is** `(1, 9)`, providing one score for each of the 9 classes.

In [ ]:
# Print the model's architecture
print(prototype_model)

# Call the helper function to visualize the data flow
print("\n--- Tracing Data Flow ---")
print_data_flow(prototype_model)

# Training the Model
With your model defined and the data pipeline prepared, you're ready to set up the training process. This involves initializing a loss function to measure your model's error and an optimizer to update its weights based on that error.

### Initialize Loss Function and Optimizer

Before starting the training loop, you'll define two key components:

* You'll use <code>[nn.CrossEntropyLoss](https://docs.pytorch.org/docs/stable/generated/torch.nn.CrossEntropyLoss.html)</code>. This is the standard loss function for multi-class classification tasks as it's designed to measure the error when a model has to choose one class from several possibilities.
* You'll use the <code>[Adam](https://docs.pytorch.org/docs/stable/generated/torch.optim.Adam.html)</code> optimizer. This is a popular and efficient algorithm that updates the model's weights to minimize the loss.

In [ ]:
# Loss function
loss_function = nn.CrossEntropyLoss()

# Optimizer for the prototype model
optimizer_prototype = optim.Adam(prototype_model.parameters(),lr=0.001)

# The Training Loop

* Next, you'll define the `training_loop` function. This function encapsulates the entire process of training and validating your model over multiple epochs.

In [ ]:
def training_loop(model, train_loader, val_loader, loss_function, optimizer, num_epochs, device):
  """
  Trains and validates a PyTorch neural network model.

  Args:
      model: The neural network model to be trained.
      train_loader: DataLoader for the training dataset.
      val_loader: DataLoader for the validation dataset.
      loss_function: The loss function to use for training.
      optimizer: The optimization algorithm.
      num_epochs: The total number of epochs to train for.
      device: The device (e.g., 'cpu' or 'cuda') to run the training on.

  Returns:
      A tuple containing:
      - The trained model.
      - A list of metrics [train_losses, val_losses, val_accuracies].
  """

  # Move the model to the specified device (CPU or GPU)
  model.to(device)

  # Initialize lists to store training and validation metrics
  train_losses = []
  val_losses = []
  val_accuracies = []

  # Print a message indicating the start of the training process
  print("--- Training Started ---")

  # Loop over the specified number of epochs
  for epoch in range(num_epochs):
    # Set the model to training mode
    model.train()

    #Initialize running loss for the current epoch
    running_loss = 0.0

    # Iterate over batches of data in the training loader
    for images, labels in train_loader:
      # Move images and labels to the specified device
      images, labels = images.to(device), labels.to(device)

      # Clear the gradients of all optimized variables
      optimizer.zero_grad()

      # Perform a forward pass to get model outputs
      outputs = model(images)

      # Calculate the loss
      loss = loss_function(outputs, labels)

      # Perform a backward pass to compute gradients
      loss.backward()

      # Update the model parameters
      optimizer.step()

      # Accumulate the training loss for the batch
      running_loss += loss.item() * images.size(0)

    # Calculate the average training loss for the epoch
    epoch_loss = running_loss / len(train_loader.dataset)

    # Append the epoch loss to the list of training losses
    train_losses.append(epoch_loss)

    # Set the model to evaluation mode
    model.eval()

    # Initialize running validation loss and correct predictions count
    running_val_loss  = 0.0
    correct = 0
    total = 0

    # Disable gradient calculations for validation
    with torch.no_grad():
      # Iterate over batches of data in the validation loader
      for images, labels in val_loader:
        # Move images and labels to the specified device
        images, labels = images.to(device), labels.to(device)

        # Perfomr a forward pass to get model outputs
        outputs = model(images)

        # Calculate the validation loss for the batch
        val_loss = loss_function(outputs, labels)

        # Accumulate the validation loss
        running_val_loss += val_loss.item() * images.size(0)

        # Get the predicted class labels
        _, predicted = torch.max(outputs, 1)

        # Update the total number of samples
        total += labels.size(0)

        # Update the number of correct predictions
        correct += (predicted == labels).sum().item()

    # Calculate the average validation loss for the epoch
    epoch_val_loss = running_val_loss / len(val_loader.dataset)

    # Append the epoch validation loss to the list
    val_losses.append(epoch_val_loss)


    # Calculate the validation accuracy for the epoch
    epoch_accuracy = 100.0 * correct / total

    # Append the epoch accuracy to the list
    val_accuracies.append(epoch_accuracy)

    # Print the metrics for the current epoch
    print(f"Epoch [{epoch+1}/{num_epochs}], Train Loss: {epoch_loss:.4f}, Val Loss: {epoch_val_loss:.4f}, Val Accuracy: {epoch_accuracy:.2f}%")

  #Print a message indicating the end of the training process
  print("--- Finished Training ---")

  # Consolidate all metrics into a single list
  metrics = [train_losses, val_losses, val_accuracies]

  # Return the trained model and the collected metrics
  return model, metrics







With all the components in place, you're ready to start training.

* Run the `training_loop` function with your prototype model (for 9 classes), its corresponding data loaders, the loss function, and the optimizer.
* You'll train for `15 epochs`, and the function will return the trained model along with the collected performance metrics.
* After training is complete, you'll use the `plot_training_metrics` helper function to visualize the training and validation loss, along with the validation accuracy.

In [ ]:
# Start the training process by calling the training loop function
trained_proto_model, training_metrics_proto = training_loop(
    model=prototype_model,
    train_loader=train_loader_proto,
    val_loader=val_loader_proto,
    loss_function=loss_function,
    optimizer=optimizer_prototype,
    num_epochs=15,
    device=device
)

# Visualize the training metrics for the full model
print("\n--- Training Plots ---\n")
plot_training_metrics(training_metrics_proto)

<br>

Excellent work! The prototype model is trained, and the results look very promising. Achieving a validation accuracy of over **75%** on the 9-class subset is a great result and confirms that your CNN architecture is well-suited for this task.

This successful prototype gives you the green light to move forward with the next phase: training a full-scale model on all 15 classes for the butterfly house. But before you do, it's helpful to perform one last qualitative check to see how your model "thinks."

### Visualizing Predictions

While the plots show your model's overall performance, looking at individual predictions provides a more intuitive feel for its strengths and weaknesses. You can now use a helper function to see your model in action, visualizing its predictions on random images from the validation set. This will show you concrete examples of where it succeeds and where it might be making mistakes.

In [ ]:
visualise_predictions(
    model=trained_proto_model,
    data_loader=val_loader_proto,
    device=device,
    grid=(3, 3)
)